In [1]:
# ============================================================
#  CELL 1 — Dataset Loading + Preprocessing + Augmentation
#  Project : RVCE EL — AMD vs Normal Fundus Classifier (ViT-B/16)
#  Framework: PyTorch + timm
# ============================================================

import os
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


# ── 1. Config ─────────────────────────────────────────────────
BASE = (
    "/kaggle/input/datasets/abuhurer"
    "/macular-degeneration-disease-armd"
    "/Macular Degeneration Disease Dataset"
)

CFG = dict(
    TRAIN_AMD    = f"{BASE}/train/amd",
    TRAIN_NORMAL = f"{BASE}/train/normal",
    VAL_AMD      = f"{BASE}/val/amd",
    VAL_NORMAL   = f"{BASE}/val/normal",

    CLASS_NAMES  = ["Normal", "AMD"],   # 0 = Normal, 1 = AMD
    IMG_SIZE     = 224,
    CLAHE_CLIP   = 2.0,
    CLAHE_TILE   = (8, 8),

    BATCH_SIZE   = 32,
    NUM_WORKERS  = 2,
    SEED         = 42,
)

torch.manual_seed(CFG["SEED"])


# ── 2. Sanity check ───────────────────────────────────────────
print("\n[Sanity] Checking dataset mount …")
for key in ("TRAIN_AMD", "TRAIN_NORMAL", "VAL_AMD", "VAL_NORMAL"):
    d = pathlib.Path(CFG[key])
    if not d.exists():
        raise FileNotFoundError(f"Directory not found: {d}")
    files = list(d.iterdir())
    print(f"  {key:15s} → {len(files):4d} files  |  sample: {files[0].name}")


# ── 3. Preprocessing helpers ──────────────────────────────────
def apply_clahe(bgr_img: np.ndarray) -> np.ndarray:
    """BGR uint8 → CLAHE grayscale stacked to 3-channel uint8."""
    gray     = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(
                   clipLimit=CFG["CLAHE_CLIP"],
                   tileGridSize=CFG["CLAHE_TILE"]
               )
    enhanced = clahe.apply(gray)
    return np.stack([enhanced, enhanced, enhanced], axis=-1)   # (H,W,3)


def center_crop(img: np.ndarray, crop_size: int) -> np.ndarray:
    """Square center crop with reflection padding if needed."""
    h, w = img.shape[:2]
    if h < crop_size or w < crop_size:
        pad_h = max(0, crop_size - h)
        pad_w = max(0, crop_size - w)
        img = cv2.copyMakeBorder(
            img,
            pad_h // 2, pad_h - pad_h // 2,
            pad_w // 2, pad_w - pad_w // 2,
            cv2.BORDER_REFLECT_101
        )
        h, w = img.shape[:2]
    top  = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return img[top : top + crop_size, left : left + crop_size]


def preprocess_image(path: str) -> np.ndarray:
    """
    CLAHE grayscale + center crop + resize.
    Returns (224, 224, 3) uint8 — used in Dataset and Flask API.
    """
    img = cv2.imread(path)
    img = apply_clahe(img)
    img = center_crop(img, CFG["IMG_SIZE"])
    img = cv2.resize(
              img,
              (CFG["IMG_SIZE"], CFG["IMG_SIZE"]),
              interpolation=cv2.INTER_AREA
          )
    return img   # uint8


# ── 4. Albumentations pipelines ───────────────────────────────

# ViT pretrained on ImageNet → use ImageNet mean/std for normalisation
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=0,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.4
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    A.CoarseDropout(
        max_holes=4,
        max_height=20,
        max_width=20,
        fill_value=0,
        p=0.15
    ),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),   # (H,W,C) uint8 → (C,H,W) float32
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


# ── 5. Dataset class ──────────────────────────────────────────
class FundusDataset(Dataset):
    def __init__(self, amd_dir: str, normal_dir: str, transform=None):
        self.transform = transform
        self.paths, self.labels = [], []

        for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
            for p in pathlib.Path(normal_dir).glob(ext):
                self.paths.append(str(p))
                self.labels.append(0)
            for p in pathlib.Path(amd_dir).glob(ext):
                self.paths.append(str(p))
                self.labels.append(1)

        print(f"  Normal : {self.labels.count(0):4d}")
        print(f"  AMD    : {self.labels.count(1):4d}")
        print(f"  Total  : {len(self.paths):4d}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img   = preprocess_image(self.paths[idx])   # (224,224,3) uint8
        label = self.labels[idx]

        if self.transform:
            img = self.transform(image=img)["image"]  # → (3,224,224) float32

        return img, torch.tensor(label, dtype=torch.float32)


# ── 6. Build datasets + dataloaders ──────────────────────────
print("\n[Train]")
train_dataset = FundusDataset(CFG["TRAIN_AMD"], CFG["TRAIN_NORMAL"], train_transform)

print("\n[Val]")
val_dataset = FundusDataset(CFG["VAL_AMD"], CFG["VAL_NORMAL"], val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True,
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")


# ── 7. Sanity check ───────────────────────────────────────────
imgs, lbls = next(iter(train_loader))
print(f"\n[Sanity] Batch shape  : {imgs.shape}")       # (32, 3, 224, 224)
print(f"[Sanity] dtype        : {imgs.dtype}")         # torch.float32
print(f"[Sanity] value range  : [{imgs.min():.3f}, {imgs.max():.3f}]")
print(f"[Sanity] Labels       : {lbls.numpy()}")
print(f"[Sanity] AMD in batch : {int(lbls.sum())}")
print(f"[Sanity] Norm in batch: {int((lbls==0).sum())}")


PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4

[Sanity] Checking dataset mount …
  TRAIN_AMD       →  782 files  |  sample: 1269.jpg
  TRAIN_NORMAL    →  600 files  |  sample: 11232_right.jpeg
  VAL_AMD         →  488 files  |  sample: 1269.jpg
  VAL_NORMAL      →  200 files  |  sample: image291.png

[Train]
  Normal :  600
  AMD    :  782
  Total  : 1382

[Val]
  Normal :  200
  AMD    :  488
  Total  :  688

Train batches : 44
Val   batches : 22

[Sanity] Batch shape  : torch.Size([32, 3, 224, 224])
[Sanity] dtype        : torch.float32
[Sanity] value range  : [-2.118, 2.640]
[Sanity] Labels       : [0. 1. 0. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1.
 1. 0. 0. 0. 1. 0. 1. 1.]
[Sanity] AMD in batch : 21
[Sanity] Norm in batch: 11


In [2]:
# ============================================================
#  CELL 2 — Model: ViT-B/16 pretrained (timm) + training loop
#  Project : RVCE EL — AMD vs Normal Fundus Classifier
# ============================================================

import timm
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

print(f"timm : {timm.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")


# ── 1. Load pretrained ViT-B/16 ───────────────────────────────
# timm ships with ImageNet pretrained ViT-B/16 weights — no URL needed
# num_classes=0 removes the classification head → outputs (B, 768)

print("\nLoading vit_base_patch16_224 …")
backbone = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=0,        # removes head → raw 768-dim CLS token output
)
print("Loaded.")


# ── 2. Binary classification model ───────────────────────────
class ViTBinaryClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.backbone.requires_grad_(False)   # freeze base

        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        features = self.backbone(x)    # (B, 768)
        return self.head(features)     # (B, 1)


model = ViTBinaryClassifier(backbone).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params     : {total:,}")
print(f"Trainable params : {trainable:,}")   # ~200K (head only)


# ── 3. Loss, optimizer, scheduler ────────────────────────────
criterion = nn.BCELoss()
optimizer = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
)
scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

print("\nModel compiled and ready.")
print(f"Loss      : BCELoss")
print(f"Optimizer : Adam  lr=1e-3")
print(f"Scheduler : CosineAnnealingLR  T_max=20")


# ── 4. Training loop ──────────────────────────────────────────
EPOCHS    = 20
SAVE_PATH = "/kaggle/working/best_vit_model.pth"

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)   # (B,1)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * imgs.size(0)
        preds          = (outputs >= 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += imgs.size(0)

    scheduler.step()

    train_loss /= train_total
    train_acc   = train_correct / train_total

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.to(DEVICE).unsqueeze(1)

            outputs    = model(imgs)
            loss       = criterion(outputs, labels)
            val_loss  += loss.item() * imgs.size(0)
            preds      = (outputs >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total  += imgs.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    # ── Save best ─────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        saved = "✓ saved"
    else:
        saved = ""

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f}  acc: {val_acc:.4f} {saved}"
    )

print(f"\nBest val accuracy : {best_val_acc:.4f}")
print(f"Model saved to    : {SAVE_PATH}")

timm : 1.0.25
Device : cuda

Loading vit_base_patch16_224 …


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loaded.

Total params     : 85,995,777
Trainable params : 197,121

Model compiled and ready.
Loss      : BCELoss
Optimizer : Adam  lr=1e-3
Scheduler : CosineAnnealingLR  T_max=20


[ WARN:0@29.605] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@29.605] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@30.991] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@30.991] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@37.690] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@37.690] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@37.778] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@37.778] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@39.729] global 

Epoch 01/20 | Train loss: 0.5788  acc: 0.6845 | Val loss: 0.3647  acc: 0.8299 ✓ saved


...

Epoch 18/20 | Train loss: 0.3243  acc: 0.8473 | Val loss: 0.2485  acc: 0.9012 


[ WARN:0@967.794] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@967.794] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@968.104] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@968.104] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@969.410] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@969.411] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@969.473] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@969.474] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@970.097

Epoch 19/20 | Train loss: 0.3218  acc: 0.8466 | Val loss: 0.2532  acc: 0.9012 


[ WARN:0@1009.028] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@1009.028] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@1009.766] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@1009.766] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@1010.925] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@1010.925] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0@1011.333] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 11 (0xb) encountered
[ WARN:0@1011.333] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 59932 (0xea1c) encountered
[ WARN:0

Epoch 20/20 | Train loss: 0.3175  acc: 0.8480 | Val loss: 0.2532  acc: 0.9026 ✓ saved

Best val accuracy : 0.9026
Model saved to    : /kaggle/working/best_vit_model.pth
